# Register one metric view

Inner task of the `register_each_mv` forEach (one iteration per manifest entry). Reads the YAML, substitutes `${source_catalog}`, `${source_schema}`, `${target_catalog}`, `${target_schema}` with `mv_yaml.render_ddl`, and runs `CREATE OR REPLACE VIEW <catalog>.<schema>.<mv_name> WITH METRICS LANGUAGE YAML AS $$...$$`.

**Widgets:** `bundle_root`, `yaml_path` (relative to `bundle_root`), `mv_name`, `source_catalog`, `source_schema`, `target_catalog`, `target_schema`. All are required; the job passes the target schema from the `ontology_schema` resource.

In [0]:
%pip install --upgrade databricks-sdk pyyaml
dbutils.library.restartPython()

In [0]:
WIDGETS = ["bundle_root", "yaml_path", "mv_name", "source_catalog", "source_schema", "target_catalog", "target_schema"]
for w in WIDGETS:
    dbutils.widgets.text(w, "")
args = {w: dbutils.widgets.get(w).strip() for w in WIDGETS}

missing = [k for k, v in args.items() if not v]
if missing:
    raise ValueError(f"Required widget(s) not set: {missing}")
for k, v in args.items():
    print(f"{k}={v}")

In [0]:
import importlib
import json
import re
import sys
import types
from pathlib import Path


def load_sdk(root: str):
    """Import mv_yaml and manifest from <root>/src/genie_ontology_modeled_knowledge_sdk_vendored/.

    The vendored files are verbatim copies of src/genie_ontology_modeled_knowledge_sdk/ (they import
    each other as genie_ontology_modeled_knowledge_sdk.*), so the vendored folder is registered as
    the genie_ontology_modeled_knowledge_sdk package. Falls back to an installed genie_ontology_modeled_knowledge_sdk.
    """
    vendored = Path(root) / "src" / "genie_ontology_modeled_knowledge_sdk_vendored"
    if (vendored / "mv_yaml.py").is_file():
        pkg = types.ModuleType("genie_ontology_modeled_knowledge_sdk")
        pkg.__path__ = [str(vendored)]
        sys.modules["genie_ontology_modeled_knowledge_sdk"] = pkg
    try:
        return (importlib.import_module("genie_ontology_modeled_knowledge_sdk.mv_yaml"),
                importlib.import_module("genie_ontology_modeled_knowledge_sdk.manifest"))
    except ImportError as exc:
        raise ImportError(
            f"genie_ontology_modeled_knowledge_sdk not found: expected the vendored library in {vendored} "
            "(scaffold_project copies it in; see src/genie_ontology_modeled_knowledge_sdk_vendored/README.md)"
        ) from exc


mv_yaml, manifest = load_sdk(args["bundle_root"])

In [0]:
_IDENT = re.compile(r"^[A-Za-z_][A-Za-z0-9_]*$")


def quote(name: str) -> str:
    """Plain identifiers as-is; anything else backtick-quoted (no backticks allowed)."""
    if _IDENT.match(name):
        return name
    if "`" in name:
        raise ValueError(f"identifier {name!r} must not contain backticks")
    return f"`{name}`"


mv_name = args["mv_name"]
if not _IDENT.match(mv_name):
    raise ValueError(f"mv_name {mv_name!r} must be an identifier ([A-Za-z_][A-Za-z0-9_]*)")

root = Path(args["bundle_root"]).resolve()
yaml_file = (root / args["yaml_path"]).resolve()
if root not in yaml_file.parents or not yaml_file.is_file():
    raise ValueError(f"yaml_path {args['yaml_path']!r} is not a file inside {root}")

params = {k: quote(args[k]) for k in ("source_catalog", "source_schema", "target_catalog", "target_schema")}
fq_name = f"{params['target_catalog']}.{params['target_schema']}.{mv_name}"

try:
    ddl = mv_yaml.render_ddl(yaml_file.read_text(), fq_name, params)
except ValueError as exc:
    raise ValueError(f"{args['yaml_path']}: cannot render DDL: {exc}") from exc
print(ddl)

In [0]:
try:
    spark.sql(ddl)
except Exception as exc:
    raise RuntimeError(f"Registering {fq_name} from {args['yaml_path']} failed: {exc}") from exc

columns = [r["col_name"] for r in spark.sql(f"DESCRIBE TABLE {fq_name}").collect()
           if r["col_name"] and not r["col_name"].startswith("#")]
print(f"Registered {fq_name}: {len(columns)} column(s)")

In [0]:
# catalog / schema: the resolved target (same values targets.resolve_target writes to resolved_target.json),
# so the run output also carries where the metric views were registered.
summary = {
    "name": mv_name,
    "fq_name": fq_name,
    "catalog": args["target_catalog"],
    "schema": args["target_schema"],
    "yaml_path": args["yaml_path"],
    "columns": len(columns),
    "status": "registered",
}
dbutils.jobs.taskValues.set(key="summary", value=summary)
dbutils.notebook.exit(json.dumps(summary))